# D200, Problem Set 2: Discrete Choice Models

Due: 19 February 2026 [here](https://classroom.github.com/a/Jraqcm5s) in
groups of up to 2.

Stefan Bucher

This problem set will review classification as discussed in the lecture
through the lens of discrete choice modeling, a classically used method
in economics.

The problem set uses the
[choice-learn](https://github.com/artefactory/choice-learn) package, see
[here](https://medium.com/artefact-engineering-and-data-science/modeling-customers-decisions-in-python-with-the-choice-learn-package-37752cb7932e)
for more background.
<!-- alternatives: PyLogit, Biogeme, torch-choice, Statsmodels, scikit-learn -->

# Problem 1: The Conditional Logit Model

Discrete choice models are built on the **Random Utility Maximization
(RUM)** framework. A decision-maker chooses the alternative with the
highest utility from a set of available options. The utility of
alternative $j$ for individual $i$ is:

$$U_{ij} = V_{ij} + \varepsilon_{ij}$$

where $V_{ij}$ is the **systematic (observable) utility** and
$\varepsilon_{ij}$ is a **random error term** capturing unobserved
factors.

The **Conditional Logit** model assumes:

1.  Utility is linear in attributes:
    $V_{ij} = \sum_k \beta_{ik} \cdot x_{jk}$
2.  Errors are i.i.d. Type I Extreme Value (Gumbel) distributed

The probability of individual $i$ choosing alternative $j$ from choice
set $\mathcal{A}$ is then given by

$$P_{ij} = \frac{\exp\left(\sum_k \beta_{ik} \cdot x_{jk}\right)}{\sum_{a \in \mathcal{A}} \exp\left(\sum_k \beta_{ik} \cdot x_{ak}\right)}$$

## The ModeCanada Dataset

We’ll work with the **ModeCanada** dataset, which contains
transportation choices for intercity trips between Montréal and Toronto.
This is a classic dataset in choice modeling research.

**(1a)** Load the ModeCanada dataset and explore its structure:

In [2]:
from choice_learn.datasets import load_modecanada
transport_df = load_modecanada(as_frame=True)
print(f"Dataset shape: {transport_df.shape}")
display(transport_df.head(8))
transport_df['alt'].unique()

Dataset shape: (15520, 11)


,case,alt,choice,dist,cost,ivt,ovt,freq,income,urban,noalt
0,1,train,0,83,28.25,50,66,4,45.0,0,2
1,1,car,1,83,15.77,61,0,0,45.0,0,2
2,2,train,0,83,28.25,50,66,4,25.0,0,2
3,2,car,1,83,15.77,61,0,0,25.0,0,2
4,3,train,0,83,28.25,50,66,4,70.0,0,2
5,3,car,1,83,15.77,61,0,0,70.0,0,2
6,4,train,0,83,28.25,50,66,4,70.0,0,2
7,4,car,1,83,15.77,61,0,0,70.0,0,2


array(['train', 'car', 'bus', 'air'], dtype=object)

The data is in **long format**: each row represents one alternative
within a choice situation. Key columns:

-   `case`: identifies each choice situation (one traveler’s decision)
-   `alt`: the transportation mode (train, air, bus, car)
-   `choice`: 1 if this alternative was chosen, 0 otherwise
-   `cost`, `ivt` (in-vehicle time), `ovt` (out-of-vehicle time), `freq`
    (frequency): alternative attributes
-   `income`: traveler characteristic (same across alternatives within a
    case)

Examine a single choice situation by filtering for `case == 1`. How many
alternatives were available? Which was chosen?



In [6]:
print(transport_df[transport_df['case']==1])
print('2 alternatives available: train and car. car was the chosen one')

   case    alt  choice  dist   cost  ivt  ovt  freq  income  urban  noalt
0     1  train       0    83  28.25   50   66     4    45.0      0      2
1     1    car       1    83  15.77   61    0     0    45.0      0      2
2 alternatives available: train and car. car was the chosen one


**(1b)** The `ChoiceDataset` is choice-learn’s core data structure. It
organizes:

-   **Choices**: which alternative was selected
-   **Items features**: attributes that vary by alternative (cost, time,
    etc.)
-   **Shared features**: attributes that are constant across
    alternatives (income, etc.)

Convert the DataFrame to a `ChoiceDataset`:

In [7]:
from choice_learn.data import ChoiceDataset

canada_dataset = ChoiceDataset.from_single_long_df(
    df=transport_df,
    items_id_column="alt",           # identifies each alternative
    choices_id_column="case",         # identifies each choice situation
    choices_column="choice",          # indicates which was chosen
    shared_features_columns=["income"],  # traveler characteristics
    items_features_columns=["cost", "freq", "ovt", "ivt"],  # alternative attributes
    choice_format="one_zero"
)

print(canada_dataset.summary())

%=====================================================================%
%%% Summary of the dataset:
%=====================================================================%
Number of items: 4
Number of choices: 4324
%=====================================================================%
 Shared Features by Choice:
 1 shared features
 with names: (['income'],)


 Items Features by Choice:
4 items features 
 with names: (['cost', 'freq', 'ovt', 'ivt'],)
%=====================================================================%



## Model Specification

**(1c)** The key modeling decision is specifying the utility function.
For ModeCanada, consider:

$$U_{ij} = \beta^{inter}_j + \beta^{cost} \cdot \text{cost}_j + \beta^{freq} \cdot \text{freq}_j + \beta^{ovt} \cdot \text{ovt}_j + \beta^{ivt}_j \cdot \text{ivt}_j + \beta^{income}_j \cdot \text{income}_i$$

**Note the subscripts:**

-   $\beta^{cost}$, $\beta^{freq}$, $\beta^{ovt}$ are **shared**
    coefficients (same effect for all modes)
-   $\beta^{ivt}_j$, $\beta^{income}_j$, $\beta^{inter}_j$ are
    **alternative-specific** (different for each mode)

Why might we want different coefficients for in-vehicle time across
modes? (Think about the experience of traveling by train vs. car
vs. plane.)


*We might want different coefficients for in-vehicle time across modes as we need to be able to consider the differences in experiences on different modes of transport. That is when a person is thinking about time in a plane vs time in a car, these are two different experiences and have different considerations, which we need to account for.*

**(1d)** Implement and fit the Conditional Logit model from (1c) using
choice-learn’s `ConditionalLogit` class. Use the utility specification
above, with `optimizer="lbfgs"` and `get_report=True`.

**Hints:**

-   Use `add_shared_coefficient()` for coefficients that are the same
    across all alternatives, and `add_coefficients()` for
    alternative-specific ones.
-   For alternative-specific constants (intercept, income), you must
    normalize one alternative to zero. Why?



In [22]:
from choice_learn.models import ConditionalLogit
modecanada_model = ConditionalLogit(optimizer='lbfgs',)
# alternative-specific coefficients, dropping one category to avoid multicollinearity
modecanada_model.add_coefficients(feature_name='intercept', items_indexes=[1,2,3])
modecanada_model.add_coefficients(feature_name='ivt', items_indexes=[1,2,3])
modecanada_model.add_coefficients(feature_name='income', items_indexes=[1,2,3])
# shared coefficients
modecanada_model.add_shared_coefficient(feature_name='cost', items_indexes=[0,1,2,3])
modecanada_model.add_shared_coefficient(feature_name='freq', items_indexes=[0,1,2,3])
modecanada_model.add_shared_coefficient(feature_name='ovt', items_indexes=[0,1,2,3])

modecanada_model.fit(canada_dataset, get_report=True)

Using L-BFGS optimizer, setting up .fit() function
Using L-BFGS optimizer, setting up .fit() function


{'train_loss': [<tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: shape=(), dtype=float32, numpy=4.5749583>,
  <tf.Tensor: shape=(), dtype=float32, numpy=39.344704>,
  <tf.Tensor: sha

**(1e)** Interpret the estimated coefficients:

1.  What is the sign of $\beta^{cost}$? Does this make economic sense?
2.  Compare the intercepts across modes. Which mode has the highest
    “baseline” utility?
3.  How do the income coefficients vary? What does this tell us about
    mode choice and income?




In [27]:
modecanada_model.report
# 1. beta cost is equal to -0.00726. this makes economic sense as with cost, utility decreases
# 2. air has the highest 'baseline' utility (the order of variables is train, car, bus, air; train is the baseline)
# 3. all income coefficients are negative. magnitude-wise from lowest to highest: bus, car, air. relative to train, this means that higher income prefer train over all other modes. they avoid air the most, and avoid car more than bus. 

,Coefficient Name,Coefficient Estimation,Std. Err,z_value,P(.>z)
0,beta_intercept_0,1.119490,0.271082,4.129715,3.632133e-05
1,beta_intercept_1,2.776829,0.176100,15.768449,0.000000e+00
2,beta_intercept_2,3.258909,0.243878,13.362862,0.000000e+00
3,beta_ivt_0,-0.011648,0.001619,-7.194599,6.266099e-13
4,beta_ivt_1,-0.015804,0.000650,-24.331341,0.000000e+00
5,beta_ivt_2,-0.006260,0.000549,-11.410392,0.000000e+00
6,beta_income_0,-0.064509,0.004914,-13.127463,0.000000e+00
7,beta_income_1,-0.025757,0.002804,-9.184946,0.000000e+00
8,beta_income_2,-0.038797,0.003324,-11.672598,0.000000e+00
9,beta_cost,-0.007260,0.001733,-4.190245,2.786538e-05


**(1f)** **Price Elasticity** measures how choice probabilities change
with price. For the logit model:

$$\eta_{jj} = \frac{\partial P_{ij}}{\partial p_j} \cdot \frac{p_j}{P_{ij}} = \beta^{cost} \cdot p_j \cdot (1 - P_{ij})$$

This is the **own-price elasticity**. Compute it for the car alternative
at the mean values.


In [36]:
# beta cost
beta_cost =  -0.00726009
# mean cost of car
mean_cost_car = transport_df[transport_df['alt']=='car']['cost'].mean()
# predicted probabilities
pr = modecanada_model.predict_probas(canada_dataset)
pr_car_mean = pr.numpy()[:,1].mean()

# elasticity
elasticity = beta_cost * mean_cost_car * (1 - pr_car_mean)
print(elasticity)


-0.46122396015944145


# Problem 2: RUMnet — Neural Network Choice Models

The Conditional Logit assumes utility is *linear* in attributes.
**RUMnet** (Aouad & Désir, 2022) relaxes this assumption using neural
networks while maintaining the RUM framework.

**(2a)** For this problem, we’ll use the more complex [**Expedia** hotel
booking dataset](https://www.kaggle.com/c/expedia-personalized-sort).
First download `train.csv` from Kaggle and save it to your Python
environment’s `choice_learn/datasets/data/expedia.csv` (if the path is
wrong, `choice_learn` will tell you the exact location in a
`FileNotFoundError`).

Load the dataset using
`load_expedia(as_frame=False, preprocessing="rumnet")`, keep only the
first 5000 choices for speed, and split 80/20 into training and test
sets. Explore the dataset structure — how many choices, items, and
features does it have? What do the choice set sizes look like?



In [5]:
from choice_learn.datasets import load_expedia
expedia_df = load_expedia(as_frame = False, preprocessing="rumnet")
# had to trim the dataset using the terminal for the load to work


/home/notei/miniconda3/envs/ML-in-Economics/lib/python3.11/site-packages/choice_learn/datasets/expedia.py:168: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  choices = expedia_df.groupby("srch_id").apply(lambda x: x.booking_bool.argmax())
/home/notei/miniconda3/envs/ML-in-Economics/lib/python3.11/site-packages/choice_learn/datasets/expedia.py:260: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[contex

In [ ]:
expedia_small = expedia_df[:5000]
train = expedia_small[:4000]
test = expedia_small[4000:]
print(expedia_small.summary())

%=====================================================================%
%%% Summary of the dataset:
%=====================================================================%
Number of items: 39
Number of choices: 5000
%=====================================================================%
 Shared Features by Choice:
 13 shared features
 with names: (['srch_length_of_stay', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'booking_window', 'random_bool', 'day_of_week', 'month', 'hour'], ['site_id', 'visitor_location_country_id', 'srch_destination_id'])


 Items Features by Choice:
11 items features 
 with names: (['prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'position', 'promotion_flag', 'orig_destination_distance', 'log_price'], ['prop_country_id'])
%=====================================================================%



**(2b)** Write down a sensible model specification for the Conditional
Logit model for the Expedia dataset, for example using the hotel
features: log(price), star rating, review, whether the hotel is a brand,
location desirability scores. You may also want to include hotel fixed
effects. Fit your model and report the cross-entropy loss on the test
data using TensorFlow’s `tf.keras.losses.CategoricalCrossentropy`.



$$U_{ij} = \beta^{price} \cdot \text{log(price)} + \beta^{stars} \cdot \text{stars} + \beta^{review} \cdot \text{review} + \beta^{brand} \cdot \text{brand} + \beta^{location1} \cdot \text{location1} + \beta^{location2} \cdot \text{location2} 

In [23]:
from choice_learn.models import ConditionalLogit
import tensorflow as tf

all_items = list(range(39))

model = ConditionalLogit(optimizer='Adam', lr=0.001, epochs=15, batch_size=128)
model.add_shared_coefficient(feature_name='log_price', items_indexes=all_items)
model.add_shared_coefficient(feature_name='prop_starrating', items_indexes=all_items)
model.add_shared_coefficient(feature_name='prop_review_score', items_indexes=all_items)
model.add_shared_coefficient(feature_name='prop_brand_bool', items_indexes=all_items)
model.add_shared_coefficient(feature_name='prop_location_score1', items_indexes=all_items)
model.add_shared_coefficient(feature_name='prop_location_score2', items_indexes=all_items)

model.fit(train, get_report=True)

# cross-entropy loss
test_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=False)(
    y_pred=model.predict_probas(test),
    y_true=tf.one_hot(test.choices, 39)
)
print(test_loss)

  0%|          | 0/15 [00:00<?, ?it/s]

Epoch 14 Train Loss 2.6888: 100%|██████████| 15/15 [00:50<00:00,  3.38s/it]


tf.Tensor(2.7311647, shape=(), dtype=float32)


**(2c)** Display the resulting parameter estimates and interpret them.
What is the sign of the price coefficient? Which features matter most?



In [25]:
model.report

,Coefficient Name,Coefficient Estimation,Std. Err,z_value,P(.>z)
0,beta_log_price,-0.292047,0.008160,-35.788200,0.000000e+00
1,beta_prop_starrating,0.263747,0.030096,8.763672,0.000000e+00
2,beta_prop_review_score,0.214954,0.033806,6.358419,2.038403e-10
3,beta_prop_brand_bool,0.169627,0.055723,3.044101,2.333770e-03
4,beta_prop_location_score1,0.197952,0.027341,7.240093,4.485301e-13
5,beta_prop_location_score2,0.358066,0.058740,6.095752,1.089243e-09


*The sign of the price coefficient is negative. Featuers that matter most are price by far. Another important feature is on location score 2*

**(2d)** Now fit the **RUMnet** model shipped with `choice_learn` to the
Expedia dataset. The dataset has 46 product features and 84 customer
features. Report the cross-entropy loss on the test data and compare it
to the Conditional Logit.



In [35]:
from choice_learn.models import RUMnet

model_args = {
    "num_products_features": 13,
    "num_customer_features": 20,
    "width_eps_x": 10,
    "depth_eps_x": 3,
    "heterogeneity_x": 5,
    "width_eps_z": 10,
    "depth_eps_z": 3,
    "heterogeneity_z": 5,
    "width_u": 10,
    "depth_u": 3,
    "optimizer": "Adam",
    "lr": 0.001,
    "logmin": 1e-10,
    "label_smoothing": 0.02,
    "callbacks": [],
    "epochs": 15,
    "batch_size": 128,
    "tol": 1e-5,
}

rumnet = RUMnet(**model_args)
rumnet.instantiate()
rumnet.fit(train, val_dataset=test)

Epoch 14 Train Loss 2.4073: 100%|██████████| 15/15 [02:55<00:00, 11.68s/it]


{'train_loss': [<tf.Tensor: shape=(), dtype=float32, numpy=3.1465771>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.828142>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.6711252>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.6176786>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.5862715>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.565514>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.544442>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.5230207>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.5010424>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.4758701>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.4595308>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.4281025>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.423927>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.4041648>,
  <tf.Tensor: shape=(), dtype=float32, numpy=2.4073277>],
 'val_loss': [3.0060182,
  2.7820005,
  2.7123618,
  2.6827054,
  2.663521,
  2.6429513,
  2.623973,
  2.602913,
  2.5852244,
  2.5716

In [36]:
test_loss = tf.keras.losses.CategoricalCrossentropy(from_logits=False)(
    y_pred=rumnet.predict_probas(test),
    y_true=tf.one_hot(test.choices, 39)
)
print(float(test_loss))

2.3416638374328613


*RUMnet loss ~2.34, ConditionalLogit's - ~2.73. RUMnet is doing better.*

**(2e)** Discuss: What are the tradeoffs between Conditional Logit and
RUMnet?

*ConditionalLogit is faster and more interpretable, however it has a lower predictive performance. RUMnet captures non-linearity therefore is better at predicting.*